In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
import nltk
nltk.download('punkt'),  nltk.download('punkt_tab')

# Train

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from itertools import chain
import click
import os

from ruamel.yaml import YAML

# Загрузка данных

In [ ]:
from src.train import load_external_data, enc_classes

# получаем данные митра
conf = YAML().load(open('params.yaml'))

df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=True)

# на самом деле 208 train тут уже есть - синтетика
df['split'] = df['split'].fillna('tr')

In [ ]:
from src.funcs import set_seed
from src.spec_nn_funcs import train_bert
from src.aug_sent import add_aug_sents

conf = YAML().load(open('params.yaml'))
set_seed(conf['seed'])
# до TextModelClass, где нейронка инициализируется
from src.spec_nn_funcs import TextDFDataset, TextModelClass, train_eval_bert

conf = YAML().load(open('params.yaml'))

conf_dop = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))


model_bert, loss_bert_d, thresh_l, _, _, _, (p_tr_micro, r_tr_micro, f1_tr_micro, p_tr_macro, r_tr_macro, f1_tr_macro) = train_bert(df, mlb, conf, conf_dop, target_col= 'labels', thresh_space_l=[])

In [ ]:
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf = YAML().load(open('params.yaml'))
conf_dop = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))
# NEED?
set_seed(conf['seed'])

# чтобы новый конф работал вместо старого в функции
conf_ttp['feat_gen'] = conf_ttp['feat_gen_ttp'] 
conf_ttp['seed'] = conf['seed']
conf_ttp['use_only_proc'] = conf['use_only_proc']

mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])

df = add_aug_sents(df, conf_ttp, conf_dop['nn_ttp']['maxlen'])



df['target'] = mlb_ttp.transform(df['ttp']).tolist()

In [ ]:
conf = YAML().load(open('params.yaml'))

conf['feat_gen'] = conf_ttp['feat_gen_ttp']
conf['train_eval_model'] = conf_ttp['train_eval_model_ttp']


conf_dop['nn'] = conf_dop['nn_ttp']
conf_dop['nn_bert'] = conf_dop['nn_bert_ttp']


model_bert_ttp, loss_bert_ttp_d, thresh_ttp_l, _, _, _, (p_tr_micro, r_tr_micro, f1_tr_micro, p_tr_macro, r_tr_macro, f1_tr_macro) = train_bert(df, mlb_ttp, conf, conf_dop, target_col= 'ttp', 
                                                                                                                                                thresh_space_l=np.arange(0.001, 1, 0.002))

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])
data['labels'] = data['labels'].map(lambda x: eval(x))
data['origin_labels'] = data['origin_labels'].map(lambda x: eval(x))
data['origin_ttp'] = data['origin_ttp'].map(lambda x: eval(x))


In [ ]:
df.shape, data.shape, df.index, data.index

## тест идентичности основных полей после загрузки

In [ ]:

data[['sentence', 'labels', 'url', 'par_name', 'is_proc']].explode('labels').compare(df[['sentence', 'labels', 'url', 'par_name', 'is_proc']].explode('labels'))

In [ ]:
data[['sentence', 'origin_ttp', 'url', 'par_name', 'is_proc']].explode('origin_ttp').reset_index(drop=True).\
compare(df[['sentence', 'origin_ttp', 'url', 'par_name', 'is_proc']].explode('origin_ttp').reset_index(drop=True))


In [ ]:
data[['sentence', 'origin_labels', 'url', 'par_name', 'is_proc']].explode('origin_labels').reset_index(drop=True).\
compare(df[['sentence', 'origin_labels', 'url', 'par_name', 'is_proc']].explode('origin_labels').reset_index(drop=True))


# код по добавлению энкодинга

In [ ]:
df_s = df.copy()

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

use_rare_ttp = False
mlb_ttp = MultiLabelBinarizer()


if use_rare_ttp:
    ttp_counts_thresh = conf['prep_text']['ttp_counts_thresh']
    ttp_l = df['origin_labels'].explode('origin_labels').value_counts().loc[lambda x: x>ttp_counts_thresh].index.tolist()    
    mlb_ttp.fit([[c] for c in ttp_l+['rare']])
    df['ttp'] = df['origin_labels'].map(lambda x: [it if it in ttp_l else 'rare' for it in x] )
else:
    ttp_l = df['origin_labels'].explode('origin_labels').value_counts().index.tolist()    
    mlb_ttp.fit([[c] for c in ttp_l])
    df['ttp'] = df['origin_labels']

In [ ]:
CLASSES = df.explode('labels')['labels'].dropna().unique()

mlb = MultiLabelBinarizer(classes=CLASSES)
mlb.fit([[c] for c in CLASSES])

df['target'] = mlb.transform(df['labels']).tolist()


df['target_ttp'] = mlb_ttp.transform(df['ttp']).tolist()

# joblib.dump(mlb, conf['prep_text']['mlb_fn'])
# joblib.dump(mlb_ttp, conf['prep_text']['ttp_mlb_fn'])

In [ ]:

sub_l = df['origin_ttp'].explode('origin_ttp').value_counts().loc[lambda x: x>0].index.tolist()

mlb_split = MultiLabelBinarizer()
mlb_split.fit([[c] for c in sub_l+['rare']])

df['origin_ttp_enc'] = mlb_split.transform(df['origin_ttp'].map(lambda x: [it if it in sub_l else 'rare' for it in x] )).tolist()

- train:
    - набор текстов:
        - предварительная обработка специальная:
            - добавление chatgpt для обучения
            - митре + tram + cyberthreats + chatgrpt + sentence_bert_gen
      
        - предварительная обработка общая:
            - токенизация
            - формирование датасетов и лоадеров
        
        - обучение 2 моделей 
        - генетика поверх нескольких прогнозов

    
- predict:
    - набор текстов:
        - предварительная обработка общая

## manual

# Predict